In [1]:
from src.constants import DATA_PATH
from pathlib import Path

In [12]:
full_time = 83500
full_time_month = full_time / 12

loss_nov = 9/30*full_time_month*0.8 # 9 days in Noember where I should have been full-time but was paid 20%
loss_march = 11/31*full_time_month*0.2 # 11 days in March where I should have bee 40% but was only 20%
loss_jul = 28/31*full_time_month*0.6 # 28 days in July where I should have been 100% but was only 40%
print(f"Full time: {full_time:.2f} CHF")
print(f"Full time per month: {full_time_month:.2f} CHF")
print(f"Loss in November: {loss_nov:.2f} CHF")
print(f"Loss in March: {loss_march:.2f} CHF")
print(f"Loss in July: {loss_jul:.2f} CHF")
print(f"Total loss: {loss_nov + loss_march + loss_jul:.2f} CHF")


Full time: 83500.00 CHF
Full time per month: 6958.33 CHF
Loss in November: 1670.00 CHF
Loss in March: 493.82 CHF
Loss in July: 3770.97 CHF
Total loss: 5934.78 CHF


In [13]:
# Period 1 November to 31 July
full_time_Google = 130000
full_time_month_Google = full_time_Google / 12

win_google = 19/52*full_time_Google*0.8 + 15/52*full_time_Google*0.6 # 19 weeks at 80% and 15 weeks at 60%
win_eth = 5*full_time_month*0.2 + 4*full_time_month*0.4 # 4 months at 20% and 4 months at 40%

print(f"Win at Google: {win_google:.2f} CHF")
print(f"Win at ETH: {win_eth:.2f} CHF")
print(f"Total win: {win_google + win_eth:.2f} CHF")

normal_salary = 9 * full_time_month
print(f"Normal salary for 9 months: {normal_salary:.2f} CHF")

win_total = win_google + win_eth - normal_salary
print(f"Total win compared to normal salary: {win_total:.2f} CHF")

Win at Google: 60500.00 CHF
Win at ETH: 18091.67 CHF
Total win: 78591.67 CHF
Normal salary for 9 months: 62625.00 CHF
Total win compared to normal salary: 15966.67 CHF


In [2]:
original_folder = Path('/scratch4/odietrich/git/xBD_Sentinel/data/raw/xbd_dataset')
new_folder = DATA_PATH / 'xbd_s12' / 'xbd'
num_workers = 4
overwrite = False

In [7]:
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path
from typing import Tuple

from osgeo import gdal
import rasterio
from rasterio.crs import CRS
from rasterio.transform import from_bounds
from tqdm import tqdm

from src.data.metadata import load_metadata
from src.utils.time import timeit
from src.utils.geometry import reproject_geo
from src.constants import XBD_S12_PATH
from src.data.create_aligned_vrt import create_aligned_vrt_file

df_meta = load_metadata()

row = df_meta.itertuples().__next__()

uid = row.Index
# Calculate corrected spatial info
true_bounds = reproject_geo(row.geometry, "EPSG:4326", row.best_utm).bounds
dst_crs = row.best_utm

# Determine tier folder mapping (xBD specific logic)
tier = "tier1" if row.xbd_tier == "train" else row.xbd_tier

tasks = []
for period in ["pre", "post"]:
    src_path = original_folder / tier / "images" / f"{uid}_{period}_disaster.tif"
    dst_path = new_folder / f"{uid}_{period}_disaster.vrt"  # Saved as .vrt
    print(src_path, dst_path)
    print(src_path.exists())
    print(dst_path.exists())

    create_aligned_vrt_file(src_path, dst_path, true_bounds, dst_crs)

/scratch4/odietrich/git/xBD_Sentinel/data/raw/xbd_dataset/tier1/images/guatemala-volcano_00000000_pre_disaster.tif /scratch/odietrich/git/xbd-s12/data/xbd_s12/xbd/guatemala-volcano_00000000_pre_disaster.vrt
True
False
/scratch4/odietrich/git/xBD_Sentinel/data/raw/xbd_dataset/tier1/images/guatemala-volcano_00000000_post_disaster.tif /scratch/odietrich/git/xbd-s12/data/xbd_s12/xbd/guatemala-volcano_00000000_post_disaster.vrt
True
False


/scratch/odietrich/git/xbd-s12/xbds12-env/lib/python3.12/site-packages/osgeo/gdal.py:606: FutureWarning: Neither gdal.UseExceptions() nor gdal.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(


In [10]:
from src.utils.time import timeit
from src.constants import XBD_S12_PATH
from pathlib import Path
from src.data.metadata import load_metadata
import json
import numpy as np
from shapely.wkt import loads as wkt_loads
import rioxarray as rxr
import cv2
from src.data.create_masks import mask_for_polygon

DAMAGE_DICT = {
    "no-damage": 1,
    "minor-damage": 2,
    "major-damage": 3,
    "destroyed": 4,
    "un-classified": 5,  # eg under clouds
}

def create_raster_mask(json_path: Path, out_fp: Path, nodata_value: int = 6):
    # adapted from https://github.com/PaulBorneP/Xview2_Strong_Baseline/blob/master/legacy/create_masks.py


    # Load the json file and transform to a 1024x1024 mask
    data = json.load(open(json_path))
    mask = np.zeros((1024, 1024), dtype="uint8")
    for feat in data["features"]["xy"]:
        poly = wkt_loads(feat["wkt"])
        subtype = feat["properties"]["subtype"]
        _mask = mask_for_polygon(poly)
        mask[_mask > 0] = DAMAGE_DICT[subtype]

    # Add nodata based on images (eg if the original image is cut)
    uid = out_fp.stem.split("_mask")[0]
    fp_pre = XBD_S12_PATH / "xbd" / f"{uid}_pre_disaster.tif"
    fp_post = XBD_S12_PATH / "xbd" / f"{uid}_post_disaster.tif"
    img_pre = rxr.open_rasterio(fp_pre)
    img_post = rxr.open_rasterio(fp_post)

    # Find nodata pixels (are there any valid pixels that are fully black??)
    mask_pre = (img_pre == 0).all(dim="band").values
    mask_post = (img_post == 0).all(dim="band").values
    mask_nodata = (mask_pre + mask_post).astype(int)  # Final mask for nodata values

    # Use one of the images to get the geotransform and crs
    raster = img_post[0]
    raster.rio.set_nodata(0)
    raster.values = np.where(mask_nodata, nodata_value, mask)

    # Save mask
    out_fp.parent.mkdir(parents=True, exist_ok=True)
    raster.rio.to_raster(out_fp, compress="zstd")

In [11]:
uid = 'guatemala-volcano_00000006'

folder = XBD_S12_PATH / 'masks'
folder.mkdir(exist_ok=True, parents=True)
df_meta = load_metadata()
df_meta.head(10)

,disaster,disaster_type,peril,xbd_tier,event_split,best_utm,N_intact,N_minor,N_major,N_destroyed,...,s2_cs_post,s1_date_pre,s1_date_post,s1_orbit_pre,s1_orbit_post,s1_direction_pre,s1_direction_post,s1_ids_pre,s1_ids_post,geometry
xbd_uid,,,,,,,,,,,,,,,,,,,,,
guatemala-volcano_00000000,guatemala-volcano,volcano,volcano,train,test,EPSG:32615,10,0,0,0,...,0.832290,2018-02-06,2018-06-24,26,26,DESCENDING,DESCENDING,S1B_IW_GRDH_1SDV_20180206T115329_20180206T1153...,S1A_IW_GRDH_1SDV_20180624T115419_20180624T1154...,"POLYGON ((-90.8132 14.39158, -90.81325 14.3870..."
guatemala-volcano_00000001,guatemala-volcano,volcano,volcano,train,test,EPSG:32615,0,4,0,0,...,0.858505,2018-02-06,2018-06-24,26,26,DESCENDING,DESCENDING,S1B_IW_GRDH_1SDV_20180206T115329_20180206T1153...,S1A_IW_GRDH_1SDV_20180624T115419_20180624T1154...,"POLYGON ((-90.81316 14.39612, -90.8132 14.3915..."
guatemala-volcano_00000002,guatemala-volcano,volcano,volcano,train,test,EPSG:32615,0,0,0,1,...,0.884884,2018-02-06,2018-06-24,26,26,DESCENDING,DESCENDING,S1B_IW_GRDH_1SDV_20180206T115329_20180206T1153...,S1A_IW_GRDH_1SDV_20180624T115419_20180624T1154...,"POLYGON ((-90.83303 14.42975, -90.83308 14.425..."
guatemala-volcano_00000003,guatemala-volcano,volcano,volcano,test,test,EPSG:32615,0,2,0,1,...,0.893014,2018-02-06,2018-06-24,26,26,DESCENDING,DESCENDING,S1B_IW_GRDH_1SDV_20180206T115329_20180206T1153...,S1A_IW_GRDH_1SDV_20180624T115419_20180624T1154...,"POLYGON ((-90.83294 14.43882, -90.83299 14.434..."
guatemala-volcano_00000004,guatemala-volcano,volcano,volcano,hold,test,EPSG:32615,2,6,8,4,...,0.888752,2018-02-06,2018-06-24,26,26,DESCENDING,DESCENDING,S1B_IW_GRDH_1SDV_20180206T115329_20180206T1153...,S1A_IW_GRDH_1SDV_20180624T115419_20180624T1154...,"POLYGON ((-90.82838 14.4297, -90.82842 14.4251..."
guatemala-volcano_00000005,guatemala-volcano,volcano,volcano,test,test,EPSG:32615,0,0,0,0,...,0.889637,2018-02-06,2018-06-24,26,26,DESCENDING,DESCENDING,S1B_IW_GRDH_1SDV_20180206T115329_20180206T1153...,S1A_IW_GRDH_1SDV_20180624T115419_20180624T1154...,"POLYGON ((-90.83751 14.44794, -90.83756 14.443..."
guatemala-volcano_00000006,guatemala-volcano,volcano,volcano,train,test,EPSG:32615,97,0,0,0,...,0.860822,2018-02-06,2018-06-24,26,26,DESCENDING,DESCENDING,S1B_IW_GRDH_1SDV_20180206T115329_20180206T1153...,S1A_IW_GRDH_1SDV_20180624T115419_20180624T1154...,"POLYGON ((-90.80411 14.3688, -90.80416 14.3642..."
guatemala-volcano_00000007,guatemala-volcano,volcano,volcano,train,test,EPSG:32615,9,0,0,0,...,0.850537,2018-02-06,2018-06-24,26,26,DESCENDING,DESCENDING,S1B_IW_GRDH_1SDV_20180206T115329_20180206T1153...,S1A_IW_GRDH_1SDV_20180624T115419_20180624T1154...,"POLYGON ((-90.82363 14.43873, -90.82368 14.434..."
guatemala-volcano_00000008,guatemala-volcano,volcano,volcano,train,test,EPSG:32615,0,0,0,0,...,0.844595,2018-02-06,2018-06-24,26,26,DESCENDING,DESCENDING,S1B_IW_GRDH_1SDV_20180206T115329_20180206T1153...,S1A_IW_GRDH_1SDV_20180624T115419_20180624T1154...,"POLYGON ((-90.81871 14.46591, -90.81876 14.461..."


In [12]:
original_folder = Path('/scratch4/odietrich/git/xBD_Sentinel/data/raw/xbd_dataset')
overwrite = False
for row in df_meta.itertuples():
    uid = row.Index
    if uid != 'guatemala-volcano_00000006':
        continue
    out_fp = folder / f"{uid}_mask.tif"

    tier = 'tier1' if row.xbd_tier == 'train' else row.xbd_tier
    json_fp = original_folder / tier / "labels" / f"{uid}_post_disaster.json" # always post
    create_raster_mask(json_fp, out_fp, nodata_value=6)
    break

In [17]:
import geopandas as gpd
from src.constants import XBD_S12_PATH
meta = gpd.read_file(XBD_S12_PATH / "xbd_s12_metadata.geojson")

meta = meta.reset_index(drop=True).rename_axis("hdf5_idx").reset_index()
meta.head()

,hdf5_idx,xbd_uid,disaster,disaster_type,peril,xbd_tier,event_split,best_utm,N_intact,N_minor,...,s2_cs_post,s1_date_pre,s1_date_post,s1_orbit_pre,s1_orbit_post,s1_direction_pre,s1_direction_post,s1_ids_pre,s1_ids_post,geometry
0,0,guatemala-volcano_00000000,guatemala-volcano,volcano,volcano,train,test,EPSG:32615,10,0,...,0.832290,2018-02-06,2018-06-24,26,26,DESCENDING,DESCENDING,S1B_IW_GRDH_1SDV_20180206T115329_20180206T1153...,S1A_IW_GRDH_1SDV_20180624T115419_20180624T1154...,"POLYGON ((-90.8132 14.39158, -90.81325 14.3870..."
1,1,guatemala-volcano_00000001,guatemala-volcano,volcano,volcano,train,test,EPSG:32615,0,4,...,0.858505,2018-02-06,2018-06-24,26,26,DESCENDING,DESCENDING,S1B_IW_GRDH_1SDV_20180206T115329_20180206T1153...,S1A_IW_GRDH_1SDV_20180624T115419_20180624T1154...,"POLYGON ((-90.81316 14.39612, -90.8132 14.3915..."
2,2,guatemala-volcano_00000002,guatemala-volcano,volcano,volcano,train,test,EPSG:32615,0,0,...,0.884884,2018-02-06,2018-06-24,26,26,DESCENDING,DESCENDING,S1B_IW_GRDH_1SDV_20180206T115329_20180206T1153...,S1A_IW_GRDH_1SDV_20180624T115419_20180624T1154...,"POLYGON ((-90.83303 14.42975, -90.83308 14.425..."
3,3,guatemala-volcano_00000003,guatemala-volcano,volcano,volcano,test,test,EPSG:32615,0,2,...,0.893014,2018-02-06,2018-06-24,26,26,DESCENDING,DESCENDING,S1B_IW_GRDH_1SDV_20180206T115329_20180206T1153...,S1A_IW_GRDH_1SDV_20180624T115419_20180624T1154...,"POLYGON ((-90.83294 14.43882, -90.83299 14.434..."
4,4,guatemala-volcano_00000004,guatemala-volcano,volcano,volcano,hold,test,EPSG:32615,2,6,...,0.888752,2018-02-06,2018-06-24,26,26,DESCENDING,DESCENDING,S1B_IW_GRDH_1SDV_20180206T115329_20180206T1153...,S1A_IW_GRDH_1SDV_20180624T115419_20180624T1154...,"POLYGON ((-90.82838 14.4297, -90.82842 14.4251..."


In [20]:
col_dates = [c for c in meta.columns if "date" in c]
for col in col_dates:
    # back to string
    meta[col] = meta[col].dt.strftime("%Y-%m-%d")

In [21]:
meta[col_dates].dtypes

xbd_date_pre     object
xbd_date_post    object
s2_date_pre      object
s2_date_post     object
s1_date_pre      object
s1_date_post     object
dtype: object

In [24]:
meta.loc[meta.xbd_tier.isin(["tier1", "tier3"]), "split"] = "train"
meta.loc[meta.xbd_tier == "test", "split"] = "test"
meta = meta[meta.xbd_tier != "hold"].copy()

In [26]:
meta.loc[meta.xbd_tier.isin(["train", "tier3"]), "split"] = "train"
meta.loc[meta.xbd_tier == "test", "split"] = "test"
meta = meta[meta.xbd_tier != "hold"].copy()

In [27]:
meta

,hdf5_idx,xbd_uid,disaster,disaster_type,peril,xbd_tier,event_split,best_utm,N_intact,N_minor,...,s1_date_pre,s1_date_post,s1_orbit_pre,s1_orbit_post,s1_direction_pre,s1_direction_post,s1_ids_pre,s1_ids_post,geometry,split
0,0,guatemala-volcano_00000000,guatemala-volcano,volcano,volcano,train,test,EPSG:32615,10,0,...,2018-02-06,2018-06-24,26,26,DESCENDING,DESCENDING,S1B_IW_GRDH_1SDV_20180206T115329_20180206T1153...,S1A_IW_GRDH_1SDV_20180624T115419_20180624T1154...,"POLYGON ((-90.8132 14.39158, -90.81325 14.3870...",train
1,1,guatemala-volcano_00000001,guatemala-volcano,volcano,volcano,train,test,EPSG:32615,0,4,...,2018-02-06,2018-06-24,26,26,DESCENDING,DESCENDING,S1B_IW_GRDH_1SDV_20180206T115329_20180206T1153...,S1A_IW_GRDH_1SDV_20180624T115419_20180624T1154...,"POLYGON ((-90.81316 14.39612, -90.8132 14.3915...",train
2,2,guatemala-volcano_00000002,guatemala-volcano,volcano,volcano,train,test,EPSG:32615,0,0,...,2018-02-06,2018-06-24,26,26,DESCENDING,DESCENDING,S1B_IW_GRDH_1SDV_20180206T115329_20180206T1153...,S1A_IW_GRDH_1SDV_20180624T115419_20180624T1154...,"POLYGON ((-90.83303 14.42975, -90.83308 14.425...",train
3,3,guatemala-volcano_00000003,guatemala-volcano,volcano,volcano,test,test,EPSG:32615,0,2,...,2018-02-06,2018-06-24,26,26,DESCENDING,DESCENDING,S1B_IW_GRDH_1SDV_20180206T115329_20180206T1153...,S1A_IW_GRDH_1SDV_20180624T115419_20180624T1154...,"POLYGON ((-90.83294 14.43882, -90.83299 14.434...",test
5,5,guatemala-volcano_00000005,guatemala-volcano,volcano,volcano,test,test,EPSG:32615,0,0,...,2018-02-06,2018-06-24,26,26,DESCENDING,DESCENDING,S1B_IW_GRDH_1SDV_20180206T115329_20180206T1153...,S1A_IW_GRDH_1SDV_20180624T115419_20180624T1154...,"POLYGON ((-90.83751 14.44794, -90.83756 14.443...",test
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10310,10310,woolsey-fire_00000873,woolsey-fire,fire,wildfire,tier3,train,EPSG:32611,0,0,...,2018-10-25,2018-11-18,71,71,DESCENDING,DESCENDING,S1A_IW_GRDH_1SDV_20181025T135223_20181025T1352...,S1A_IW_GRDH_1SDV_20181118T135222_20181118T1352...,"POLYGON ((-118.83491 34.11038, -118.83482 34.1...",train
10311,10311,woolsey-fire_00000874,woolsey-fire,fire,wildfire,tier3,train,EPSG:32611,0,0,...,2018-10-25,2018-11-18,71,71,DESCENDING,DESCENDING,S1A_IW_GRDH_1SDV_20181025T135223_20181025T1352...,S1A_IW_GRDH_1SDV_20181118T135222_20181118T1352...,"POLYGON ((-118.85033 34.11861, -118.85024 34.1...",train
10312,10312,woolsey-fire_00000875,woolsey-fire,fire,wildfire,tier3,train,EPSG:32611,0,0,...,2018-10-25,2018-11-18,71,71,DESCENDING,DESCENDING,S1A_IW_GRDH_1SDV_20181025T135223_20181025T1352...,S1A_IW_GRDH_1SDV_20181118T135222_20181118T1352...,"POLYGON ((-118.82983 34.11046, -118.82974 34.1...",train
10313,10313,woolsey-fire_00000876,woolsey-fire,fire,wildfire,tier3,train,EPSG:32611,0,0,...,2018-10-25,2018-11-18,71,71,DESCENDING,DESCENDING,S1A_IW_GRDH_1SDV_20181025T135223_20181025T1352...,S1A_IW_GRDH_1SDV_20181118T135222_20181118T1352...,"POLYGON ((-118.84525 34.11868, -118.84516 34.1...",train
